In [22]:
import pandas as pd
import numpy as np
import ast

from sklearn.model_selection import GroupShuffleSplit

PATH_CLIPS                          = "../../Data/Videos/Clips/"
PATH_ACTIONS_FILTERED               = "../../Data/Processed/actions_filtered.csv"

PATH_KEYPOINTS                      = "../../Data/Unprocessed/keypoints.csv"
PATH_METRICS                        = "../../Data/Unprocessed/metrics.csv"
PATH_ROI                            = "../../Data/Unprocessed/roi.csv"
PATH_LABEL_MAP                      = "../../Data/label_map.csv"

PATH_CLS_TRAIN                       = "../../Data/Processed/cls_train.csv"
PATH_CLS_TEST                        = "../../Data/Processed/cls_test.csv"

PATH_CLS_SEQ_TRAIN                   = "../../Data/Processed/cls_seq_train.csv"
PATH_CLS_SEQ_TEST                    = "../../Data/Processed/cls_seq_test.csv"

PATH_ROI_TRAIN                       = "../../Data/Processed/roi_train.csv"
PATH_ROI_TEST                        = "../../Data/Processed/roi_test.csv"

WINDOW_SIZE                         = 10
NUM_JOINTS                          = 17

In [11]:
df_metrics = pd.read_csv(PATH_METRICS)

total_expected_frames = df_metrics["expected"].sum()
total_actual_frames = df_metrics["actual"].sum()
total_coverage = total_actual_frames / total_expected_frames

print(df_metrics[df_metrics["expected"] != df_metrics["actual"]])
print("")

print("Expected frames: ", total_expected_frames)
print("Actual frames:   ", total_actual_frames)
print(f"Coverage:         {total_coverage*100:.2f}%")

    fencer  action_id  start_frame  end_frame  expected  actual   coverage  \
265  RIGHT        995           32         39         8       7  87.500000   
598  RIGHT        774           26         32         7       6  85.714286   

               file  
265  6/20_Right.mp4  
598   5/15_Left.mp4  

Expected frames:  16821
Actual frames:    16819
Coverage:         99.99%


In [12]:
df_keypoints = pd.read_csv(PATH_KEYPOINTS)
df_filtered = pd.read_csv(PATH_ACTIONS_FILTERED)

df_keypoints["keypoints"] = df_keypoints["keypoints"].apply(
    lambda x: [tuple(p) for p in ast.literal_eval(x)] if isinstance(x, str) else x
)

df_merged = df_keypoints.merge(df_filtered, on=["file", "fencer"], how="left")
df_merged = df_merged[
    (df_merged["frame"] >= df_merged["start_frame"]) &
    (df_merged["frame"] <= df_merged["end_frame"])
]

df_merged = df_merged[["file", "fencer", "action_id", "action", "frame", "start_frame", "end_frame", "confidence", "keypoints"]].reset_index(drop=True)
df_merged.head()

,file,fencer,action_id,action,frame,start_frame,end_frame,confidence,keypoints
0,1/10_Left.mp4,LEFT,0,OTHER_NO_ACTION,0,0,22,0.898973,"[(651.2871704101562, 605.3470458984375), (654...."
1,1/10_Left.mp4,LEFT,0,OTHER_NO_ACTION,1,0,22,0.885350,"[(652.6170654296875, 606.501953125), (655.7320..."
2,1/10_Left.mp4,LEFT,0,OTHER_NO_ACTION,2,0,22,0.867700,"[(654.3292236328125, 608.1226806640625), (657...."
3,1/10_Left.mp4,LEFT,0,OTHER_NO_ACTION,3,0,22,0.881394,"[(658.3043823242188, 607.803466796875), (661.1..."
4,1/10_Left.mp4,LEFT,0,OTHER_NO_ACTION,4,0,22,0.856014,"[(664.1746826171875, 605.3037719726562), (665...."


In [ ]:
def build_sequences(
    df,
    seq_len,
    minority_labels
):
    sequences = []

    for (video, fencer), g in df.groupby(["file", "fencer"]):
        g = g.sort_values("frame").reset_index(drop=True)

        labels = g["action"].values

        for i in range(len(g) - seq_len + 1):
            window_labels = labels[i:i+seq_len]

            has_minority = np.any(
                np.isin(window_labels, minority_labels)
            )

            has_boundary = np.any(
                window_labels[:-1] != window_labels[1:]
            )

            sequences.append({
                "file": video,
                "fencer": fencer,
                "start_idx": i,
                "has_minority": has_minority,
                "has_boundary": has_boundary,
            })

    return pd.DataFrame(sequences).reset_index(drop=True)

In [23]:
gss = GroupShuffleSplit(
    n_splits=1,
    train_size=0.8,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(df_merged, groups=df_merged['file'])
)

df_train = df_merged.iloc[train_idx]
df_test  = df_merged.iloc[test_idx]

minority_labels = ["ATTACK_FLUNGE", "ATTACK_POINT_IN_LINE", "DEFENSE_PARRY", "DEFENSE_DISTANCE_PULL"]
label_map = pd.read_csv(PATH_LABEL_MAP)

train_sequences = build_sequences(df_train, seq_len=WINDOW_SIZE, minority_labels=minority_labels, label_map=label_map)
test_sequences = build_sequences(df_test, seq_len=WINDOW_SIZE, minority_labels=minority_labels, label_map=label_map)

print(train_sequences[["has_minority", "has_boundary"]].mean())
print("")
print(test_sequences[["has_minority", "has_boundary"]].mean())

has_minority    0.124800
has_boundary    0.375531
dtype: float64

has_minority    0.120124
has_boundary    0.349633
dtype: float64


In [26]:
df_train = df_train[["file", "fencer", "frame", "action", "keypoints"]]
df_test = df_test[["file", "fencer", "frame", "action", "keypoints"]]

df_train.to_csv(PATH_CLS_TRAIN, index=False)
df_test.to_csv(PATH_CLS_TEST, index=False)

train_sequences.to_csv(PATH_CLS_SEQ_TRAIN, index=False)
test_sequences.to_csv(PATH_CLS_SEQ_TEST, index=False)

In [16]:
df_roi = pd.read_csv(PATH_ROI)

gss_roi = GroupShuffleSplit(
    n_splits=1,
    train_size=0.8,
    random_state=42
)

train_idx, test_idx = next(
    gss_roi.split(df_roi, groups=df_roi['file'])
)

df_train_roi = df_roi.iloc[train_idx]
df_test_roi  = df_roi.iloc[test_idx]

df_train_roi.to_csv(PATH_ROI_TRAIN, index=False)
df_test_roi.to_csv(PATH_ROI_TEST, index=False)